### Read the ERP location data from the Bronze layer, apply the required cleaning and standardization steps, then save the final cleaned output to the Silver layer as a Delta table.

# Initialization


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, length

# Read Bronze table



In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")


# Display the raw Bronze data first

In [0]:
# Display the raw Bronze data first to understand the file structure
# and identify the data quality issues that need cleaning in the Silver layer.
df.display()

- ============================================================

Data quality issues found in this file:
 1. Some string columns may contain leading or trailing spaces.
 2. The customer ID contains hyphens and should be standardized to match the customer master key format.
 3. The country column contains null or blank values.
 4. The country column contains inconsistent values such as US, USA, and United States.
 5. The country column also contains values such as DE and Germany, so country names should be standardized.
 6. We should check for duplicate records based on the business key.
 7. Column names need to be renamed to cleaner business-friendly names before saving the final Silver table.

============================================================

# Trimming


In [0]:
# Trimming string columns
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Validation checks before cleaning


In [0]:
# Validation checks before cleaning ERP location data
df.select(
    F.count(F.when(col("CID").isNull(), True)).alias("null_customer_id_count"),
    F.count(F.when(col("CNTRY").isNull(), True)).alias("null_country_count"),
    F.count(F.when(trim(col("CNTRY")) == "", True)).alias("blank_country_count"),
    F.count(F.when(col("CNTRY").isin("US", "USA"), True)).alias("us_country_variants_count"),
    F.count(F.when(col("CNTRY").isin("DE", "Germany"), True)).alias("germany_country_variants_count")
).display()

# Customer ID Cleanup



In [0]:
# Standardizing the customer ID to match the customer master key format
df = df.withColumn("CID", F.regexp_replace(col("CID"), "-", ""))

# Country Cleanup



In [0]:
# Standardizing country values
df = df.withColumn(
    "CNTRY",
    F.when(col("CNTRY").isin("US", "USA", "United States"), "United States")
     .when(col("CNTRY").isin("DE", "Germany"), "Germany")
     .when(col("CNTRY").isNull() | (trim(col("CNTRY")) == ""), "n/a")
     .otherwise(col("CNTRY"))
)

# Validation checks after cleaning



In [0]:
# Validation checks after cleaning ERP location data
df.select(
    F.count(F.when(col("CID").isNull(), True)).alias("null_customer_id_count"),
    F.count(F.when(~col("CID").rlike("^AW[0-9]+$"), True)).alias("invalid_customer_id_count"),
    F.count(F.when(col("CNTRY").isNull(), True)).alias("null_country_count"),
    F.count(F.when(trim(col("CNTRY")) == "", True)).alias("blank_country_count"),
    F.count(F.when(col("CNTRY") == "n/a", True)).alias("unknown_country_count")
).display()

# check duplicate record on bussiness keys in this file



In [0]:
# Check for duplicate records based on the business key
# to make sure the location rows are unique before saving to Silver.
df.groupBy("CID") \
  .count() \
  .filter(col("count") > 1) \
  .display()

### there is no dublicate 

#rename coulmns



In [0]:
# Renaming columns to cleaner business names
RENAME_MAP = {
    "CID": "customer_number",
    "CNTRY": "country"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#quick validation for df


In [0]:
# Display the cleaned dataframe for a quick validation check
df.display()

#writing silver table



In [0]:
# Writing Silver table
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customer_location")

#check table created or not



In [0]:
%sql
-- Preview the final Silver table
SELECT * FROM workspace.silver.erp_customer_location LIMIT 10